# 04. Diffusion Model — Hyperparameter Tuning with Optuna

This notebook optimises the **Conditional Diffusion Model** hyperparameters using [Optuna](https://optuna.org/) (Bayesian optimisation with pruning).

### Why Optuna instead of GridSearch?
| | GridSearch | Optuna |
|---|---|---|
| **Strategy** | Try every combination (brute force) | Learn from past trials, propose smarter next guess |
| **Trials needed** | 243+ (5 params × 3 values) | ~30–50 trials for same quality |
| **Early stopping** | ❌ Must finish every trial | ✅ Kills bad trials early (pruning) |
| **PyTorch support** | ❌ Needs sklearn wrapper | ✅ Native, first-class support |

### Hyperparameters Tuned
| Tier | Parameter | Search Range |
|---|---|---|
| **High Impact** | `learning_rate` | 1e-4 → 1e-2 (log) |
| **High Impact** | `hidden_dim` | [128, 256, 512] |
| **High Impact** | `num_timesteps` | [200, 500, 1000] |
| **High Impact** | `cfg_scale` | 1.0 → 3.0 |
| **Medium** | `batch_size` | [64, 128, 256, 512] |
| **Medium** | `cfg_dropout` | [0.05, 0.1, 0.15, 0.2] |
| **Medium** | `schedule_type` | ['cosine', 'linear'] |
| **Medium** | `time_emb_dim` | [32, 64, 128] |

In [ ]:
# =============================================================================
# IMPORTS & SETUP
# =============================================================================
import sys, os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

import optuna
from optuna.exceptions import TrialPruned

# Add parent directory to path for src imports
sys.path.append(os.path.abspath('..'))
from src.data_processing import calculate_caps, apply_capping
from src.diffusion_model import (
    ElectricityPriceDataset,
    DiffusionSchedule,
    DiffusionDenoiser,
    compute_loss,
    sample,
)
from src.utils import seed_everything

print('All imports loaded.')

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================
REGION = 'NSW1'
DATA_PATH = '../data/prices_wide_clean.parquet'
START_DATE = '2021-10-01'
N_LAGS = 6
PREDICTION_HORIZON = 6  # 6 × 5-min = 30-min ahead
TRAIN_RATIO = 0.8
RANDOM_STATE = 42
RESULTS_PATH = '../models_profiles.json'

# Optuna-specific settings
N_TRIALS = 30           # Number of Optuna trials
TUNING_EPOCHS = 30      # Reduced epochs for search (speed)
FINAL_EPOCHS = 50       # Full epochs for best config
STUDY_DB = '../results/diffusion_study.db'  # SQLite for resumable studies

# Device detection
if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')      # Apple Silicon GPU
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')     # NVIDIA GPU
else:
    DEVICE = torch.device('cpu')

seed_everything(RANDOM_STATE)

print(f'Region:          {REGION}')
print(f'Lags:            {N_LAGS}')
print(f'Horizon:         {PREDICTION_HORIZON} intervals (={PREDICTION_HORIZON*5} min)')
print(f'Split:           {TRAIN_RATIO:.0%} / {1-TRAIN_RATIO:.0%}')
print(f'Device:          {DEVICE}')
print(f'Tuning Trials:   {N_TRIALS}')
print(f'Tuning Epochs:   {TUNING_EPOCHS}')
print(f'Final Epochs:    {FINAL_EPOCHS}')

In [ ]:
# =============================================================================
# DATA LOADING (from Silver parquet) — same pipeline as 03
# =============================================================================
try:
    df_wide = pd.read_parquet(DATA_PATH)
except FileNotFoundError:
    raise FileNotFoundError("Run '00_data_preprocess.ipynb' first.")

# Ensure datetime index
if not isinstance(df_wide.index, pd.DatetimeIndex):
    df_wide.index = pd.to_datetime(df_wide.index)

df_wide = df_wide[df_wide.index >= START_DATE]

# Keep only NSW1 for this notebook
df_nsw = df_wide[[REGION]].copy()
df_nsw = df_nsw.reset_index()
df_nsw = df_nsw.rename(columns={'index': 'date_time'}) if 'date_time' not in df_nsw.columns else df_nsw

print(f'Loaded: {len(df_nsw):,} rows, range: {df_nsw["date_time"].min()} → {df_nsw["date_time"].max()}')
df_nsw.head()

In [ ]:
# =============================================================================
# CAPPING (IQR method, same as baseline)
# =============================================================================
min_caps, max_caps = calculate_caps(df_nsw, regions=[REGION])
df_capped = apply_capping(df_nsw, min_caps, max_caps, regions=[REGION])

print(f'\nCap range: ${min_caps[REGION]:.2f} to ${max_caps[REGION]:.2f}')
print(f'Values capped: {(df_nsw[REGION] != df_capped[REGION]).sum():,}')

In [ ]:
# =============================================================================
# FEATURE ENGINEERING: Lag Features + Cyclical Time Encoding
# =============================================================================
df_feat = df_capped.copy()

# --- Lag features ---
for lag in range(1, N_LAGS + 1):
    df_feat[f'{REGION}_lag{lag}'] = df_feat[REGION].shift(lag)

# --- Cyclical time encoding (from Experiment 2) ---
hour = df_feat['date_time'].dt.hour
dow  = df_feat['date_time'].dt.dayofweek
mon  = df_feat['date_time'].dt.month

df_feat['hour_sin']  = np.sin(2 * np.pi * hour / 24)
df_feat['hour_cos']  = np.cos(2 * np.pi * hour / 24)
df_feat['dow_sin']   = np.sin(2 * np.pi * dow / 7)
df_feat['dow_cos']   = np.cos(2 * np.pi * dow / 7)
df_feat['month_sin'] = np.sin(2 * np.pi * mon / 12)
df_feat['month_cos'] = np.cos(2 * np.pi * mon / 12)

# --- Target: 30 min ahead ---
df_feat['target'] = df_feat[REGION].shift(-PREDICTION_HORIZON)

# --- Clean & split ---
df_feat = df_feat.dropna().reset_index(drop=True)

feature_cols = (
    [c for c in df_feat.columns if 'lag' in c]
    + ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos']
)
X = df_feat[feature_cols].values
y = df_feat['target'].values

print(f'Features: {len(feature_cols)}')
print(f'Feature columns: {feature_cols}')
print(f'Total samples: {len(X):,}')

In [ ]:
# =============================================================================
# TRAIN / VAL / TEST SPLIT (time-based, no shuffling)
# =============================================================================
n_train = int(len(X) * TRAIN_RATIO)
n_val   = int(len(X) * 0.1)  # 10% for validation during tuning

X_train_raw, y_train_raw = X[:n_train - n_val], y[:n_train - n_val]
X_val_raw,   y_val_raw   = X[n_train - n_val:n_train], y[n_train - n_val:n_train]
X_test_raw,  y_test_raw  = X[n_train:], y[n_train:]

# Scale features (fit on train only)
feat_scaler = StandardScaler()
X_train_scaled = feat_scaler.fit_transform(X_train_raw)
X_val_scaled   = feat_scaler.transform(X_val_raw)
X_test_scaled  = feat_scaler.transform(X_test_raw)

# Scale targets (fit on train only)
target_scaler = StandardScaler()
y_train_scaled = target_scaler.fit_transform(y_train_raw.reshape(-1, 1)).flatten()
y_val_scaled   = target_scaler.transform(y_val_raw.reshape(-1, 1)).flatten()
y_test_scaled  = target_scaler.transform(y_test_raw.reshape(-1, 1)).flatten()

print(f'Train: {len(X_train_scaled):,} samples')
print(f'Val:   {len(X_val_scaled):,} samples')
print(f'Test:  {len(X_test_scaled):,} samples')

---
## Optuna Objective Function

This function is called once per trial. Optuna suggests hyperparameters, we build and train a diffusion model with them, and return the validation loss.

In [ ]:
# =============================================================================
# HELPER: Evaluate diffusion model on a dataset
# =============================================================================
def evaluate_diffusion(model, schedule, dataloader, device,
                       target_scaler, cfg_scale=1.5, n_samples=5):
    """
    Evaluate the diffusion model by sampling predictions and computing RMSE.
    Uses multiple samples per input and takes the mean (Monte Carlo estimate).
    """
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for features, targets in dataloader:
            features = features.to(device)
            targets  = targets.to(device)

            # Sample n_samples predictions and average them
            batch_preds = []
            for _ in range(n_samples):
                pred = sample(model, features, schedule, cfg_scale=cfg_scale)
                batch_preds.append(pred.squeeze(-1))

            # Mean of samples → point estimate
            mean_pred = torch.stack(batch_preds).mean(dim=0)
            all_preds.append(mean_pred.cpu().numpy())
            all_targets.append(targets.cpu().numpy())

    all_preds   = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)

    # Inverse-transform back to original scale
    all_preds_original   = target_scaler.inverse_transform(all_preds.reshape(-1, 1)).flatten()
    all_targets_original = target_scaler.inverse_transform(all_targets.reshape(-1, 1)).flatten()

    rmse = np.sqrt(mean_squared_error(all_targets_original, all_preds_original))
    return rmse

print('Helper functions defined ✓')

In [ ]:
# =============================================================================
# OPTUNA OBJECTIVE FUNCTION
# =============================================================================
def create_objective(
    X_train, y_train, X_val, y_val,
    target_scaler, device, num_epochs=30
):
    """
    Returns an Optuna objective function (closure) that trains
    a diffusion model with suggested hyperparameters.
    """
    n_features = X_train.shape[1]

    def objective(trial):
        # ── Tier 1: High Impact ──
        lr            = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
        hidden_dim    = trial.suggest_categorical('hidden_dim', [128, 256, 512])
        num_timesteps = trial.suggest_categorical('num_timesteps', [200, 500, 1000])
        cfg_scale     = trial.suggest_float('cfg_scale', 1.0, 3.0)

        # ── Tier 2: Medium Impact ──
        batch_size    = trial.suggest_categorical('batch_size', [64, 128, 256, 512])
        cfg_dropout   = trial.suggest_categorical('cfg_dropout', [0.05, 0.1, 0.15, 0.2])
        schedule_type = trial.suggest_categorical('schedule_type', ['cosine', 'linear'])
        time_emb_dim  = trial.suggest_categorical('time_emb_dim', [32, 64, 128])

        # ── Build DataLoaders ──
        train_dataset = ElectricityPriceDataset(X_train, y_train)
        val_dataset   = ElectricityPriceDataset(X_val, y_val)
        train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader    = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)

        # ── Build Schedule ──
        schedule = DiffusionSchedule(
            num_timesteps=num_timesteps,
            device=device,
            schedule_type=schedule_type
        )

        # ── Build Model ──
        model = DiffusionDenoiser(
            n_features=n_features,
            time_emb_dim=time_emb_dim,
            hidden_dim=hidden_dim,
            cfg_dropout=cfg_dropout
        ).to(device)

        optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

        # ── Training loop with pruning ──
        best_val_loss = float('inf')

        for epoch in range(num_epochs):
            # --- Train ---
            model.train()
            epoch_train_loss = 0.0
            n_batches = 0

            for features, targets in train_loader:
                features = features.to(device)
                targets  = targets.to(device).unsqueeze(1)

                loss = compute_loss(model, targets, features, schedule, use_cfg=True)

                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

                epoch_train_loss += loss.item()
                n_batches += 1

            # --- Validate ---
            model.eval()
            epoch_val_loss = 0.0
            n_val_batches = 0

            with torch.no_grad():
                for features, targets in val_loader:
                    features = features.to(device)
                    targets  = targets.to(device).unsqueeze(1)

                    val_loss = compute_loss(model, targets, features, schedule, use_cfg=False)
                    epoch_val_loss += val_loss.item()
                    n_val_batches += 1

            avg_val_loss = epoch_val_loss / n_val_batches

            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss

            scheduler.step()

            # ── Pruning: report intermediate value ──
            trial.report(avg_val_loss, epoch)
            if trial.should_prune():
                raise TrialPruned()

        return best_val_loss

    return objective

print('Objective function defined ✓')

---
## Run the Optuna Study

The study is saved to a SQLite database so it can be **resumed** if interrupted.

In [ ]:
# =============================================================================
# RUN OPTUNA STUDY
# =============================================================================
print('='*70)
print('  DIFFUSION MODEL — Optuna Hyperparameter Search')
print('='*70)
print(f'⚠ Running {N_TRIALS} trials with {TUNING_EPOCHS} epochs each.')
print(f'  Bad trials will be pruned early to save compute.')
print(f'  Study stored at: {STUDY_DB}')
print()

# Create or load study (resumable via SQLite)
study = optuna.create_study(
    study_name='diffusion_tuning',
    direction='minimize',
    storage=f'sqlite:///{STUDY_DB}',
    load_if_exists=True,
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=5,     # Let first 5 trials run fully
        n_warmup_steps=5,       # Don't prune before epoch 5
    ),
)

# Build objective
objective = create_objective(
    X_train=X_train_scaled,
    y_train=y_train_scaled,
    X_val=X_val_scaled,
    y_val=y_val_scaled,
    target_scaler=target_scaler,
    device=DEVICE,
    num_epochs=TUNING_EPOCHS,
)

# Run optimization
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print()
print('='*70)
print('  SEARCH COMPLETE')
print('='*70)
print(f'  Best trial:     #{study.best_trial.number}')
print(f'  Best val loss:  {study.best_value:.6f}')
print(f'  Best params:')
for k, v in study.best_params.items():
    print(f'    {k}: {v}')

---
## Visualize Optimization Results

In [ ]:
# =============================================================================
# VISUALIZATION: Optimization History
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Plot 1: Optimization History ---
trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
trial_numbers = [t.number for t in trials]
trial_values  = [t.value  for t in trials]

# Running best
running_best = []
current_best = float('inf')
for v in trial_values:
    current_best = min(current_best, v)
    running_best.append(current_best)

ax = axes[0]
ax.scatter(trial_numbers, trial_values, alpha=0.6, color='#3498db', label='Trial value', zorder=2)
ax.plot(trial_numbers, running_best, color='#e74c3c', linewidth=2, label='Best so far', zorder=3)
ax.set_xlabel('Trial #')
ax.set_ylabel('Validation Loss')
ax.set_title('Optimization History', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# --- Plot 2: Parameter Importance (manual bar chart) ---
ax = axes[1]
try:
    importances = optuna.importance.get_param_importances(study)
    params = list(importances.keys())
    values = list(importances.values())
    
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(params)))
    ax.barh(params, values, color=colors)
    ax.set_xlabel('Importance')
    ax.set_title('Hyperparameter Importance', fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
except Exception as e:
    ax.text(0.5, 0.5, f'Could not compute\nimportance:\n{e}',
            ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Hyperparameter Importance', fontweight='bold')

plt.suptitle('Diffusion Model — Optuna Results', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/figures/diffusion_optuna_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to: ../results/figures/diffusion_optuna_results.png')

---
## Retrain Best Model for Full Epochs

Now we take the winning hyperparameters and train for the full epoch budget.

In [ ]:
# =============================================================================
# RETRAIN WITH BEST HYPERPARAMETERS
# =============================================================================
best = study.best_params

print('='*70)
print('  RETRAINING WITH BEST HYPERPARAMETERS')
print('='*70)
for k, v in best.items():
    print(f'  {k}: {v}')
print()

seed_everything(RANDOM_STATE)

# Build final components
final_schedule = DiffusionSchedule(
    num_timesteps=best['num_timesteps'],
    device=DEVICE,
    schedule_type=best['schedule_type']
)

final_model = DiffusionDenoiser(
    n_features=len(feature_cols),
    time_emb_dim=best['time_emb_dim'],
    hidden_dim=best['hidden_dim'],
    cfg_dropout=best['cfg_dropout']
).to(DEVICE)

final_optimizer = torch.optim.AdamW(final_model.parameters(), lr=best['learning_rate'])
final_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(final_optimizer, T_max=FINAL_EPOCHS)

# Use ALL training data (train + val) for final model
X_full_train = np.concatenate([X_train_scaled, X_val_scaled])
y_full_train = np.concatenate([y_train_scaled, y_val_scaled])

final_train_dataset = ElectricityPriceDataset(X_full_train, y_full_train)
final_test_dataset  = ElectricityPriceDataset(X_test_scaled, y_test_scaled)
final_train_loader  = DataLoader(final_train_dataset, batch_size=best['batch_size'], shuffle=True)
final_test_loader   = DataLoader(final_test_dataset,  batch_size=best['batch_size'], shuffle=False)

# Training loop (full epochs, no pruning)
train_losses = []
best_model_state = None
# Use a small held-out portion for early stopping signal
best_train_loss = float('inf')

for epoch in range(FINAL_EPOCHS):
    final_model.train()
    epoch_loss = 0.0
    n_batches = 0

    for features, targets in final_train_loader:
        features = features.to(DEVICE)
        targets  = targets.to(DEVICE).unsqueeze(1)

        loss = compute_loss(final_model, targets, features, final_schedule, use_cfg=True)

        final_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(final_model.parameters(), max_norm=1.0)
        final_optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1

    avg_loss = epoch_loss / n_batches
    train_losses.append(avg_loss)

    if avg_loss < best_train_loss:
        best_train_loss = avg_loss
        best_model_state = {k: v.cpu().clone() for k, v in final_model.state_dict().items()}

    final_scheduler.step()

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'  Epoch [{epoch+1:3d}/{FINAL_EPOCHS}] | Train Loss: {avg_loss:.6f}')

# Load best weights
if best_model_state:
    final_model.load_state_dict(best_model_state)
    final_model = final_model.to(DEVICE)

print()
print(f'  Training complete. Best train loss: {best_train_loss:.6f}')

In [ ]:
# =============================================================================
# TRAINING LOSS CURVE
# =============================================================================
plt.figure(figsize=(10, 4))
plt.plot(train_losses, color='#3498db', linewidth=1.5)
plt.xlabel('Epoch')
plt.ylabel('Train Loss')
plt.title('Final Model — Training Loss', fontweight='bold')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../results/figures/diffusion_final_training_loss.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to: ../results/figures/diffusion_final_training_loss.png')

---
## Evaluate on Test Set

In [ ]:
# =============================================================================
# TEST SET EVALUATION
# =============================================================================
print('='*70)
print('  EVALUATING ON TEST SET')
print('='*70)
print('  Sampling predictions (this may take a minute)...')

test_rmse = evaluate_diffusion(
    model=final_model,
    schedule=final_schedule,
    dataloader=final_test_loader,
    device=DEVICE,
    target_scaler=target_scaler,
    cfg_scale=best['cfg_scale'],
    n_samples=10,  # More samples for final evaluation
)

print(f'\n  Test RMSE: ${test_rmse:.2f}')
print(f'\n  (For comparison, XGBoost best RMSE was $40.13)')

---
## Save Results

In [ ]:
# =============================================================================
# SAVE RESULTS TO models_profiles.json
# =============================================================================

# Load existing profiles
try:
    with open(RESULTS_PATH, 'r') as f:
        profiles = json.load(f)
except FileNotFoundError:
    profiles = {}

# Add/update Diffusion entry
profiles['Diffusion'] = {
    'best_rmse': round(test_rmse, 2),
    'best_params': best
}

# Save
with open(RESULTS_PATH, 'w') as f:
    json.dump(profiles, f, indent=4, default=str)

print(f'Results saved to: {RESULTS_PATH}')
print(json.dumps(profiles, indent=4, default=str))

In [ ]:
# =============================================================================
# SAVE BEST MODEL WEIGHTS
# =============================================================================
model_save_path = '../results/models/diffusion_best.pt'

torch.save({
    'model_state_dict': best_model_state,
    'best_params': best,
    'test_rmse': test_rmse,
    'n_features': len(feature_cols),
    'feature_cols': feature_cols,
    'feat_scaler_mean': feat_scaler.mean_.tolist(),
    'feat_scaler_scale': feat_scaler.scale_.tolist(),
    'target_scaler_mean': float(target_scaler.mean_[0]),
    'target_scaler_scale': float(target_scaler.scale_[0]),
}, model_save_path)

print(f'Model saved to: {model_save_path}')

---
## Summary

| Item | Value |
|---|---|
| Trials run | `N_TRIALS` |
| Pruned trials | See output above |
| Best val loss | See output above |
| Test RMSE | See output above |
| Model saved to | `../results/models/diffusion_best.pt` |
| Config saved to | `../models_profiles.json` |